# Experimenting

In [1]:
import pandas as pd
import os
import pytz
import statistics
from datetime import datetime
from datetime import timedelta
import numpy as np
import pthelperfunctions as helper

pd.set_option('display.max_rows', None)

#path_mob = "/gpfs/gibbs/project/jain_snigdha/shared/EM_PTOT_project/mobilization_output"
#path_ptot = "/gpfs/gibbs/project/jain_snigdha/shared/EM_PTOT_project/ot_pt_output"

ModuleNotFoundError: No module named 'pthelperfunctions'

In [2]:
#Load block data from our project output.
path = os.path.join(helper.path_out, "block_compiled_5.parquet")
block_df = pd.read_parquet(path)

#Load the CSV with the columns we want
path = os.path.join(helper.path_out, "items_to_look.csv")
items_df = pd.read_csv(path)
items_chart_df = items_df[items_df['linksto']=='chartevents']
items_proc_df = items_df[items_df['linksto']=='procedureevents']

In [3]:
#Load mimic chart events
chart_df = helper.load_data("mimic","chartevents", folder='icu')
chart_df.rename(columns={'subject_id': 'patient_id','hadm_id':'hospitalization_id'}, inplace=True)

#Filter for our cohort and needs
chart_mask = \
        chart_df['patient_id'].isin(block_df['patient_id'].astype(int)) &\
        chart_df['itemid'].isin(items_chart_df['itemid']) &\
        chart_df['value'].notna()

filtered_chart_df = chart_df[chart_mask]

path = os.path.join(helper.path_out, "chartevents_filtered.parquet")
filtered_chart_df.to_parquet(path)

#Print DF
print(f"Length: {len(filtered_chart_df)}")
print("Columns")
filtered_chart_df.dtypes
#filtered_chart_df.head(30)

Length: 6708871
Columns


patient_id              int64
hospitalization_id      int64
stay_id                 int64
caregiver_id          float64
charttime              object
storetime              object
itemid                  int64
value                  object
valuenum              float64
valueuom               object
warning               float64
dtype: object

In [1]:
import pandas as pd
import os
import pytz
import statistics
from datetime import datetime
from datetime import timedelta
import numpy as np
import pthelperfunctions as helper

pd.set_option('display.max_rows', None)

path = os.path.join(helper.path_out, "block_compiled_5.parquet")
block_df = pd.read_parquet(path)
block_df = block_df[~block_df['pt_post48_IMV']]
BIG_N = block_df['encounter_block'].nunique()

path = os.path.join(helper.path_out, "chartevents_filtered.parquet")
filtered_chart_df = pd.read_parquet(path)

#Merge with cohort data and calculate time_window
block_df['patient_id'] = block_df['patient_id'].astype(int)
merged_chart_df = pd.merge(
    filtered_chart_df[['patient_id','itemid','value','charttime']],
    block_df[['encounter_block','patient_id','block_vent_start_dttm','pt_post48_IMV']],
    on = 'patient_id',
    how = 'right'
)
merged_chart_df['charttime'] = helper.ensure_datetime_est(merged_chart_df['charttime'])
merged_chart_df['time_diff'] = merged_chart_df['charttime'] - merged_chart_df['block_vent_start_dttm']
merged_chart_df['time_window'] = merged_chart_df['time_diff'].apply(helper.classify_time_window_ext)

#Aggregate first by encounter to get a count, then by item and time_window to look at the whole cohort
chart_counts_df = merged_chart_df.groupby(by=['encounter_block','itemid','time_window'])['value'].count().reset_index()
chart_counts_df = chart_counts_df.groupby(by=['itemid','time_window'])['value'].agg(['count','min','median','max']).reset_index()
chart_counts_df['missing'] = (BIG_N - chart_counts_df['count'])*100/BIG_N
chart_counts_df['missing'] = chart_counts_df['missing'].round(1)
chart_counts_df['itemid'] = chart_counts_df['itemid'].astype(int)

In [10]:
filtered_chart_df[filtered_chart_df['itemid'].isin([229319])].head()

,patient_id,hospitalization_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
100637,10003400,20214994,32128372,8991.0,2137-03-03 20:11:00,2137-03-03 20:11:00,229319,1 - Bedrest - Only lying,1.0,None,0.0
100747,10003400,20214994,32128372,8991.0,2137-03-03 22:00:00,2137-03-03 23:16:00,229319,1 - Bedrest - Only lying,1.0,None,0.0
100779,10003400,20214994,32128372,8991.0,2137-03-04 00:00:00,2137-03-04 00:48:00,229319,1 - Bedrest - Only lying,1.0,None,0.0
100815,10003400,20214994,32128372,8991.0,2137-03-04 02:00:00,2137-03-04 05:35:00,229319,1 - Bedrest - Only lying,1.0,None,0.0
100838,10003400,20214994,32128372,8991.0,2137-03-04 04:00:00,2137-03-04 05:35:00,229319,1 - Bedrest - Only lying,1.0,None,0.0


In [6]:
#Load the CSV with the columns we want
path = os.path.join(helper.path_out, "items_to_look.csv")
items_df = pd.read_csv(path)
items_chart_df = items_df[items_df['linksto']=='chartevents']
items_chart_df['itemid'] = items_chart_df['itemid'].astype(int)
items_chart_df.head()

/tmp/ipykernel_2441657/4135664024.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  items_chart_df['itemid'] = items_chart_df['itemid'].astype(int)


,itemid,label,abbreviation,linksto,category,param_type
0,220001,Problem List,Problem List,chartevents,General,Text
1,224084,Activity,Activity,chartevents,Treatments,Text
2,224085,Assistance Device,Assistance Device,chartevents,Treatments,Text
3,224086,Activity Tolerance,Activity Tolerance,chartevents,Treatments,Text
4,224092,Range of Motion Location,ROM Location,chartevents,Treatments,Text


In [7]:
#Merge to get the names
chart_counts_df = pd.merge(
    chart_counts_df,
    items_chart_df[['itemid','label','category']],
    on='itemid',
    how='outer'
)
#Put the labels in the front
label_column = chart_counts_df.pop('label')
category_column = chart_counts_df.pop('category')
chart_counts_df.insert(0,'label',label_column)
chart_counts_df.insert(1,'category',category_column)

#Sort
time_window_order = ['past','prior24h','0-24h','24-48h','48-72h','future']
chart_counts_df['time_window'] = pd.Categorical(chart_counts_df['time_window'], categories=time_window_order, ordered=True)
chart_counts_df = chart_counts_df.sort_values(by=['category','itemid','time_window']).reset_index()


In [8]:
path = os.path.join(helper.path_out, "chartevents_counts_noPT.csv")
chart_counts_df.to_csv(path)

chartevents
category = Treatments
labels =
    Activity / Mobility (JH-HLM)
    Range of Motion Status
    Basic Mobility (AM-PAC)    Daily Activity (AM-PAC1        
Applied Cognitive (AM-PA     1    
Test Performed (AM-P          1
Treatments (AM-           1
Discharge Recommendations         Activity / Mobility (RN Daily Mobility Goal)

chartevents
category = Adm History/FHPA
labels =
    Past medical history
    Self ADL
    History of slips / falls
    Balance
    Judgement
    Use of assistive devices
    Pressure Ulcer Present

chartevents
category = Restraint/Support Systems
label = 
    History of falling (within 3 mnths)

chartevents
category = General
labels =
    Problem List

chartevents
category = MD Progress Note
label = 
    Mobilization Plan
    Sedation Goal
    Delirium
    Disposition

procedureevents
category = 3-Significant Events
Label =
    Fall
    Unplanned Line/Catheter Removal (Patient InitPAC)